In [ ]:
%reload_ext autoreload

In [ ]:
import os
import pandas as pd
import pickle

In [ ]:
folder = "remove_partial" # change this if necessary. It depends on precedent notebook.
train_set = "train_set.csv"
test_set = "test_set.csv"
calibration_set = "calibration_set.csv"

In [ ]:
def load_df(folder,data_name):
    data_path = os.path.join(folder,data_name)
    df = pd.read_csv(data_path)
    df['start_time'] = pd.to_datetime(df['start_time'])
    df['end_time'] = pd.to_datetime(df['end_time'])
    df["start"] = df["concept:name"] + "_start"
    df["start"] = df["start"].apply(lambda row: row.replace(" ", "_").lower())
    df["end"] = df["concept:name"] + "_end"
    df["end"]= df["end"].apply(lambda row: row.replace(" ", "_").lower())
    return df

def store_df(folder,data_name, df):
    data_path = os.path.join(folder,data_name)
    df.to_csv(data_path)
    
def store_file(folder,name,obj):
    file_path = os.path.join(folder,f"{name}.pickle")
    with open(file_path, 'wb') as file:
        pickle.dump(obj, file, protocol=pickle.HIGHEST_PROTOCOL)
    
def from_df_to_trace(df): 
    ccns = df["case:concept:name"].unique()
    return {ccn: from_ccn_to_trace(df,ccn) for ccn in ccns}

def from_ccn_to_trace(df,ccn):
    ddf = df[df["case:concept:name"] == ccn][["start","end","start_time","end_time"]]
    init = ddf.iloc[0]["start_time"]
    ddf["start_time"] = (ddf["start_time"] - init)
    ddf["end_time"] = (ddf["end_time"] - init)
    ddf["start_time"] = ddf["start_time"].apply(lambda data: data.total_seconds())
    ddf["end_time"] = ddf["end_time"].apply(lambda data: data.total_seconds())
    trace = list(ddf[["start_time","start"]].itertuples(index=False,name=None)) + list(ddf[["end_time","end"]].itertuples(index=False,name=None)) 
    trace = [(time,{event}) for (time,event) in trace]
    return sorted(trace, key=lambda event: event[0])

In [ ]:
# loading dataset dataset
df_train = load_df(folder,train_set)
df_test = load_df(folder,test_set)
df_calibration = load_df(folder,calibration_set)

In [ ]:
def compute_max_time_lenght(df):
    df_min = df.groupby("case:concept:name").min()
    df_max =  df.groupby("case:concept:name").max()
    return (df_max["end_time"] - df_min["start_time"]).max()

In [ ]:
# here we evaluate the maximum time length of trajectories. 
# It is used for replace +-infinity value of time robustness.
train_max_length = compute_max_time_lenght(df_train)
test_max_length = compute_max_time_lenght(df_test)
calibration_max_length = compute_max_time_lenght(df_calibration)
max_time_lenght = max(train_max_length,test_max_length,calibration_max_length).total_seconds()
print("Max time_length (sec): ", max_time_lenght )

In [ ]:
# here we generate the labelled dataset used then for model training.
from mitl.Semantics import load_formula, evaluate_boolean_semantics, evaluate_robustness_semantics

# list of properties
# for grammar of MITL please refer to paper and "mitl/MITL.g4"
property1 = load_formula(f"G[0,INF](a_create_application_end --> F[0,{24*3600}]o_sent_end)")
# property2 = load_formula(f"G[0,INF](o_sent_end --> F[0,{9*24*3600}] w_call_after_offers_end)")

properties = [property1] # you can add more propoerties at the same time.

trace_train  = from_df_to_trace(df_train)
train_properties_dict = {aid : [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_train.items()}
store_file(folder,"train_prop",train_properties_dict)

trace_test  = from_df_to_trace(df_test)
test_properties_dict = {aid: [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_test.items()}
store_file(folder,"test_prop",test_properties_dict)

trace_calibration  = from_df_to_trace(df_calibration)
calibration_properties_dict = { aid: [evaluate_robustness_semantics(trace,prop) for prop in properties] for (aid,trace) in trace_calibration.items()}
store_file(folder,"calibration_prop",calibration_properties_dict)